# 🇺🇦 Ukrainian News Text Classification
## Three Approaches: TF-IDF + LogReg | TF-IDF + SVM | XLM-RoBERTa Fine-Tuning

---

### Overview
This notebook presents a complete pipeline for **multi-class topic classification of Ukrainian news articles** using three machine learning approaches of increasing complexity:

| # | Model | Type | Expected Macro F1 |
|---|-------|------|-------------------|
| 1 | TF-IDF + Logistic Regression | Classical baseline | 0.9+ |
| 2 | TF-IDF + LinearSVC | Improved linear | 0.9+ |
| 3 | XLM-RoBERTa (fine-tuned) | Transformer | 0.95+ |

**Dataset:** [`FIdo-AI/ua-news`](https://huggingface.co/datasets/FIdo-AI/ua-news) — 150,522 Ukrainian news articles in 5 categories.

**Categories:** News | Sports | Business | Technology | Politics

**Key challenges addressed:**
- Rich Ukrainian morphology (7 noun cases)
- Domain-specific stopword removal
- Multilingual transformer fine-tuning with limited data
- Proper evaluation: accuracy, macro P/R/F1, confusion matrix

## 1. Environment Setup

In [ ]:
# Install required packages
!pip install -q datasets transformers wordcloud scikit-learn torch numpy accelerate pandas matplotlib seaborn
print('All packages installed.')

In [ ]:
import os, re, warnings, unicodedata
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from collections import Counter
from wordcloud import WordCloud

# NLP
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import (accuracy_score, classification_report,
                              confusion_matrix, ConfusionMatrixDisplay,
                              f1_score)
from sklearn.manifold import TSNE
from sklearn.decomposition import TruncatedSVD

# HuggingFace
from datasets import load_dataset
import torch
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                           TrainingArguments, Trainer, EarlyStoppingCallback)
from torch.utils.data import Dataset as TorchDataset

warnings.filterwarnings('ignore')
plt.rcParams.update({'figure.dpi': 120, 'font.family': 'DejaVu Sans'})
sns.set_theme(style='whitegrid', palette='muted')

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
print(f'GPU: {torch.cuda.get_device_name(0) if device=="cuda" else "None"}')

## 2. Dataset Loading

In [ ]:
# Load from HuggingFace Hub
hf_dataset = load_dataset("FIdo-AI/ua-news")

df_train_raw = pd.DataFrame(hf_dataset['train'])
df_test_raw  = pd.DataFrame(hf_dataset['test']) if 'test' in hf_dataset else None

# Combine all available data, then re-split
if df_test_raw is not None:
    df_all = pd.concat([df_train_raw, df_test_raw], ignore_index=True)
else:
    df_all = df_train_raw.copy()

print(f'Total samples: {len(df_all):,}')
print(f'Columns: {df_all.columns.tolist()}')
df_all.head(3)

In [ ]:
LABEL_NAMES = sorted(df_all['target'].unique())
N_CLASSES = len(LABEL_NAMES)
print(f'Classes ({N_CLASSES}): {LABEL_NAMES}')

# Encode labels
label2id = {l: i for i, l in enumerate(LABEL_NAMES)}
id2label = {i: l for l, i in label2id.items()}
df_all['label_enc'] = df_all['target'].map(label2id)
df_all.info()

## 3. Exploratory Data Analysis (EDA)

We perform a thorough EDA to understand the dataset before modelling.

In [ ]:
# Class distribution
fig, ax = plt.subplots(figsize=(10, 5))
counts = df_all['target'].value_counts().sort_values(ascending=True)
bars = ax.barh(counts.index, counts.values,
               color=sns.color_palette('muted', N_CLASSES))
for bar, val in zip(bars, counts.values):
    ax.text(val + 30, bar.get_y() + bar.get_height()/2,
            f'{val:,} ({val/len(df_all)*100:.1f}%)',
            va='center', fontsize=10)
ax.set_xlabel('Number of Samples')
ax.set_title('Class Distribution')
ax.set_xlim(0, counts.max() * 1.2)
plt.tight_layout()
plt.savefig('class_distribution.png')
plt.show()

print(f'\nImbalance ratio (max/min): {counts.max()/counts.min():.2f}')
print('Dataset is sufficiently balanced — no SMOTE required.')

In [ ]:
# Text length statistics
df_all['char_len']  = df_all['text'].str.len()
df_all['word_count'] = df_all['text'].str.split().str.len()

print('Text length statistics:')
print(df_all[['char_len', 'word_count']].describe().round(1))
print(f"\n95th percentile char_len: {df_all['char_len'].quantile(0.95):.0f}")
print(f"95th percentile word_count: {df_all['word_count'].quantile(0.95):.0f}")

In [ ]:
# Character and word length distributions
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df_all['char_len'].clip(upper=8000), bins=60,
             color='steelblue', edgecolor='white')
axes[0].axvline(df_all['char_len'].median(), color='red', ls='--',
                label=f"Median: {df_all['char_len'].median():.0f}")
axes[0].set_xlabel('Character Count'); axes[0].set_ylabel('Frequency')
axes[0].set_title('Text Length Distribution (characters)')
axes[0].legend()

axes[1].hist(df_all['word_count'].clip(upper=1000), bins=60,
             color='darkorange', edgecolor='white')
axes[1].axvline(df_all['word_count'].median(), color='red', ls='--',
                label=f"Median: {df_all['word_count'].median():.0f}")
axes[1].set_xlabel('Word Count'); axes[1].set_ylabel('Frequency')
axes[1].set_title('Text Length Distribution (words)')
axes[1].legend()

plt.tight_layout()
plt.savefig('length_distributions.png')
plt.show()

In [ ]:
# Boxplot of word count by category
fig, ax = plt.subplots(figsize=(12, 5))
order = df_all.groupby('target')['word_count'].median().sort_values().index
df_all.boxplot(column='word_count', by='target',
               ax=ax, notch=True,
               medianprops=dict(color='red', linewidth=2),
               patch_artist=True)
ax.set_xticklabels(order, rotation=20, ha='right')
ax.set_xlabel('Category'); ax.set_ylabel('Word Count')
ax.set_title('Word Count Distribution by Category')
plt.suptitle('')
plt.tight_layout()
plt.savefig('boxplot_by_category.png')
plt.show()

In [ ]:
# Top-20 words & WordClouds
# Ukrainian stopwords
UKR_STOPWORDS = set([
    'і', 'та', 'або', 'але', 'проте', 'хоча', 'адже', 'бо', 'тому', 'адже',
    'щоб', 'якщо', 'коли', 'лише', 'тільки', 'навіть', 'також', 'щодо',
    'ще', 'вже', 'може', 'мабуть',
    'в', 'на', 'з', 'до', 'від', 'по', 'за', 'у', 'про', 'для',
    'без', 'між', 'над', 'під', 'при', 'після', 'через',
    'зі', 'із',
    'він', 'вона', 'вони', 'ми', 'ви',
    'його', 'її', 'їх', 'нас', 'вас',
    'мене', 'тебе', 'себе',
    'ним', 'ній', 'нею', 'нам', 'вам',
    'що', 'як', 'це', 'те', 'той', 'цей', 'ця', 'цих',
    'того', 'цього', 'цьому',
    'які', 'який', 'яка', 'яке', 'яких',
    'є', 'де', 'хто', 'менше', 'менш', 'більш', 'більше', 'так',
    'редакція', 'видання', 'агентство', 'агентства',
    'кореспондент', 'журналіст', 'репортер',
    'прес', 'служба',
    'укрінформ', 'уніан', 'інтерфакс', 'апостроф',
    'ліга', 'страна', 'рбк', 'еспресо'
])

def simple_tokenize(text):
    tokens = re.findall(r'[а-яіїєёґa-z]+', text.lower())
    return [t for t in tokens if t not in UKR_STOPWORDS and len(t) > 2]

# Top-20 most frequent words across corpus
all_tokens = []
for txt in df_all['text'].sample(3000, random_state=SEED):
    all_tokens.extend(simple_tokenize(txt))

top20 = Counter(all_tokens).most_common(20)
words20, freq20 = zip(*top20)

fig, ax = plt.subplots(figsize=(12, 5))
ax.barh(words20[::-1], freq20[::-1],
        color=sns.color_palette('viridis', 20)[::-1])
ax.set_xlabel('Frequency'); ax.set_title('Top-20 Most Frequent Words (after stopword removal)')
plt.tight_layout()
plt.savefig('top20_words.png')
plt.show()

In [ ]:
# WordClouds per category 
palette = sns.color_palette('Set2', N_CLASSES)
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for idx, (label, ax) in enumerate(zip(LABEL_NAMES, axes)):
    subset = df_all[df_all['target'] == label]['text'].sample(
        min(500, len(df_all[df_all['target'] == label])), random_state=SEED)
    tokens = []
    for txt in subset:
        tokens.extend(simple_tokenize(txt))
    freq = Counter(tokens)
    wc = WordCloud(width=600, height=400, background_color='white',
                   max_words=80, colormap='viridis',
                   font_path=None).generate_from_frequencies(freq)
    ax.imshow(wc, interpolation='bilinear')
    ax.axis('off')
    color = [int(c*255) for c in palette[idx]]
    ax.set_title(label, fontsize=13, fontweight='bold')

fig.suptitle('Word Frequency Clouds by Category (after stopword removal)', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('wordclouds.png')
plt.show()

In [ ]:
# Average unique words per category
def unique_word_count(text):
    return len(set(simple_tokenize(text)))

df_sample = df_all.sample(2000, random_state=SEED).copy()
df_sample['unique_words'] = df_sample['text'].apply(unique_word_count)

avg_unique = df_sample.groupby('target')['unique_words'].mean().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(avg_unique.index, avg_unique.values,
       color=sns.color_palette('muted', len(avg_unique)))
ax.set_xticklabels(avg_unique.index, rotation=25, ha='right')
ax.set_ylabel('Average Unique Words')
ax.set_title('Average Unique Word Count by Category')
for i, v in enumerate(avg_unique.values):
    ax.text(i, v + 1, f'{v:.0f}', ha='center', fontsize=10)
plt.tight_layout()
plt.savefig('unique_words.png')
plt.show()

print(avg_unique.round(1))

In [ ]:
# Top-15 bigrams
def get_bigrams(text):
    tokens = simple_tokenize(text)
    return [f'{tokens[i]} {tokens[i+1]}' for i in range(len(tokens)-1)]

all_bigrams = []
for txt in df_all['text'].sample(3000, random_state=SEED):
    all_bigrams.extend(get_bigrams(txt))

top15_bg = Counter(all_bigrams).most_common(15)
bg_words, bg_freq = zip(*top15_bg)

fig, ax = plt.subplots(figsize=(12, 5))
ax.barh(bg_words[::-1], bg_freq[::-1], color='teal')
ax.set_xlabel('Frequency')
ax.set_title('Top-15 Most Frequent Bigrams')
plt.tight_layout()
plt.savefig('bigrams.png')
plt.show()

In [ ]:
# t-SNE visualization
# Subsample for speed
df_tsne = df_all.sample(3000, random_state=SEED).reset_index(drop=True)

tfidf_tsne = TfidfVectorizer(max_features=5000, min_df=2,
                              stop_words=list(UKR_STOPWORDS))
X_tsne = tfidf_tsne.fit_transform(df_tsne['text'])

# Reduce to 50 dims with SVD before t-SNE (much faster)
svd = TruncatedSVD(n_components=50, random_state=SEED)
X_svd = svd.fit_transform(X_tsne)

tsne = TSNE(n_components=2, perplexity=40, random_state=SEED)
X_2d = tsne.fit_transform(X_svd)

fig, ax = plt.subplots(figsize=(12, 8))
palette = sns.color_palette('tab10', N_CLASSES)
for i, label in enumerate(LABEL_NAMES):
    mask = df_tsne['target'] == label
    ax.scatter(X_2d[mask, 0], X_2d[mask, 1], c=[palette[i]],
               label=label, alpha=0.5, s=15)
ax.legend(title='Category', bbox_to_anchor=(1.02, 1), loc='upper left')
ax.set_title('t-SNE Visualization (TF-IDF 5K features, 3000 samples)')
ax.set_xlabel('t-SNE dim 1'); ax.set_ylabel('t-SNE dim 2')
plt.tight_layout()
plt.savefig('tsne.png')
plt.show()

### Lexical Diversity (Type-Token Ratio)
Lexical diversity represents the ratio of unique words to the total number of words (Type-Token Ratio or TTR). A higher TTR indicates a richer and more varied vocabulary, which is often characteristic of specialized or analytical texts.

In [ ]:
# Lexical Diversity (Type-Token Ratio) by category
df_sample['ttr'] = df_sample['unique_words'] / df_sample['word_count']

fig, ax = plt.subplots(figsize=(10, 5))
sns.boxplot(x='target', y='ttr', data=df_sample, palette='Set2', ax=ax)
ax.set_title('Lexical Diversity (Type-Token Ratio) by Category')
ax.set_ylabel('Type-Token Ratio')
ax.set_xlabel('Category')
plt.tight_layout()
plt.savefig('ttr_by_category.png', dpi=300)
plt.show()

print('Average TTR by category:')
print(df_sample.groupby('target')['ttr'].mean().sort_values(ascending=False).round(3))

### Average Word Length Distribution
Analyzing the average length of words can reveal structural differences in text complexity across categories. Topics involving technical terminology or formal names (e.g., Technology, Politics) may shift the distribution towards longer words.

In [ ]:
# Average word length by category
df_all['avg_word_len'] = df_all['char_len'] / df_all['word_count'].clip(lower=1)

fig, ax = plt.subplots(figsize=(12, 5))
sns.kdeplot(data=df_all, x='avg_word_len', hue='target', fill=True, common_norm=False, palette='tab10', ax=ax)
ax.set_title('Distribution of Average Word Length by Category')
ax.set_xlabel('Average Word Length (characters)')
ax.set_ylabel('Density')
ax.set_xlim(3, 10) # Typical average word lengths in Ukrainian
plt.tight_layout()
plt.savefig('avg_word_len.png', dpi=300)
plt.show()

print('Median average word length by category:')
print(df_all.groupby('target')['avg_word_len'].median().sort_values(ascending=False).round(2))

## 4. Text Preprocessing

Pipeline: normalize → lowercase → remove HTML & punctuation → tokenize → remove stopwords

In [ ]:
def preprocess_uk(text: str) -> str:
    text = unicodedata.normalize("NFC", text)
    text = text.lower()
    text = re.sub(r"http\S+|www\.\S+", " ", text)
    text = re.sub(r"[^а-щьюяєіїґa-z\s]", " ", text)
    tokens = text.split()
    tokens = [t for t in tokens if t not in UKR_STOPWORDS and len(t) > 2]
    text = " ".join(tokens)
    return text.strip()

# Demonstration
sample_text = df_all['text'].iloc[0][:200]
print('ORIGINAL:', sample_text)
print('\nPROCESSED:', preprocess_uk(sample_text))

In [ ]:
print('Preprocessing texts...')

df_all['text_processed'] = df_all['text'].apply(preprocess_uk)

print('\nDone. SAMPLE:')
print(df_all['text_processed'].iloc[0][:200])

## 5. Train / Validation / Test Split

In [ ]:
X = df_all['text_processed'].to_numpy()
X_raw = df_all['text'].to_numpy()          # raw text for transformer
y = df_all['label_enc'].to_numpy(dtype=int)

# 70% train, 15% val, 15% test — stratified
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=SEED, stratify=y)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=SEED, stratify=y_temp)

# Same split indices for raw text (for transformer)
X_raw_train, X_raw_temp, _, _ = train_test_split(
    X_raw, y, test_size=0.30, random_state=SEED, stratify=y)
X_raw_val, X_raw_test, _, _ = train_test_split(
    X_raw_temp, y_temp, test_size=0.50, random_state=SEED, stratify=y_temp)

print(f'Train:      {len(X_train):,} samples')
print(f'Validation: {len(X_val):,} samples')
print(f'Test:       {len(X_test):,} samples')

# Verify class balance
for name, y_split in [('Train', y_train), ('Val', y_val), ('Test', y_test)]:
    counts = pd.Series(y_split).value_counts(normalize=True).sort_index()
    print(f'{name}: {counts.values.round(3)}')

## 6. Model 1: TF-IDF + Logistic Regression (Baseline)

In [ ]:
print('='*60)
print('MODEL 1: TF-IDF + Logistic Regression')
print('='*60)

pipe_lr = Pipeline([
    ('tfidf', TfidfVectorizer(
        ngram_range=(1, 2),
        max_features=50_000,
        sublinear_tf=True,
        min_df=2
    )),
    ('clf', LogisticRegression(
        C=1.0,
        solver='saga',
        max_iter=1000,
        n_jobs=-1,
        random_state=SEED
    ))
])

pipe_lr.fit(X_train, y_train)

# Validation metrics
y_pred_lr_val = pipe_lr.predict(X_val)
print(f'\nValidation Accuracy: {accuracy_score(y_val, y_pred_lr_val):.4f}')
print(f'Validation Macro F1: {f1_score(y_val, y_pred_lr_val, average="macro"):.4f}')

# Test metrics
y_pred_lr = pipe_lr.predict(X_test)
acc_lr = accuracy_score(y_test, y_pred_lr)
f1_lr  = f1_score(y_test, y_pred_lr, average='macro')
print(f'\nTest Accuracy:  {acc_lr:.4f}')
print(f'Test Macro F1:  {f1_lr:.4f}')
print('\n--- Classification Report ---')
print(classification_report(y_test, y_pred_lr,
                             target_names=[id2label[i] for i in range(N_CLASSES)]))

In [ ]:
# Confusion matrix - Model 1
fig, ax = plt.subplots(figsize=(9, 7))
cm_lr = confusion_matrix(y_test, y_pred_lr)
disp = ConfusionMatrixDisplay(cm_lr, display_labels=[id2label[i] for i in range(N_CLASSES)])
disp.plot(ax=ax, colorbar=True, cmap='Blues', xticks_rotation=30)
ax.set_title('Confusion Matrix: TF-IDF + Logistic Regression')
plt.tight_layout()
plt.savefig('cm_logreg.png')
plt.show()

## 7. Model 2: TF-IDF + LinearSVC

In [ ]:
print('='*60)
print('MODEL 2: TF-IDF + LinearSVC')
print('='*60)

# Grid search for optimal C parameter
from sklearn.model_selection import cross_val_score

best_c, best_score = 0.5, 0
tfidf_vec = TfidfVectorizer(ngram_range=(1,2), max_features=50_000,
                             sublinear_tf=True, min_df=2)
X_train_tfidf = tfidf_vec.fit_transform(X_train)
X_val_tfidf   = tfidf_vec.transform(X_val)
X_test_tfidf  = tfidf_vec.transform(X_test)

print('Searching optimal C for LinearSVC...')
for c in [0.1, 0.5, 1.0, 2.0]:
    svc = LinearSVC(C=c, max_iter=2000, random_state=SEED)
    svc.fit(X_train_tfidf, y_train)
    score = f1_score(y_val, svc.predict(X_val_tfidf), average='macro')
    print(f'  C={c:.1f} → Val Macro F1={score:.4f}')
    if score > best_score:
        best_score, best_c = score, c

print(f'\nBest C: {best_c}')

# Final model with best C
svc_final = LinearSVC(C=best_c, max_iter=2000, random_state=SEED)
svc_final.fit(X_train_tfidf, y_train)

y_pred_svc = svc_final.predict(X_test_tfidf)
acc_svc = accuracy_score(y_test, y_pred_svc)
f1_svc  = f1_score(y_test, y_pred_svc, average='macro')

print(f'\nTest Accuracy:  {acc_svc:.4f}')
print(f'Test Macro F1:  {f1_svc:.4f}')
print('\n--- Classification Report ---')
print(classification_report(y_test, y_pred_svc,
                             target_names=[id2label[i] for i in range(N_CLASSES)]))

In [ ]:
# Confusion matrix - Model 2
fig, ax = plt.subplots(figsize=(9, 7))
cm_svc = confusion_matrix(y_test, y_pred_svc)
disp = ConfusionMatrixDisplay(cm_svc, display_labels=[id2label[i] for i in range(N_CLASSES)])
disp.plot(ax=ax, colorbar=True, cmap='Oranges', xticks_rotation=30)
ax.set_title('Confusion Matrix: TF-IDF + LinearSVC')
plt.tight_layout()
plt.savefig('cm_svc.png')
plt.show()

## 8. Model 3: XLM-RoBERTa Fine-Tuning

We fine-tune `FacebookAI/xlm-roberta-base` — a multilingual transformer trained on 100 languages including Ukrainian.

> **Note:** Requires GPU. Set Kaggle accelerator to **GPU P100** or **T4**.

In [ ]:
MODEL_NAME = 'FacebookAI/xlm-roberta-base'
MAX_LEN = 512

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class NewsDataset(TorchDataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        self.texts  = texts
        self.labels = labels
        self.tok    = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tok(
            str(self.texts[idx]),
            truncation=True,
            padding='max_length',
            max_length=self.max_len,
            return_tensors='pt'
        )
        return {
            'input_ids':      enc['input_ids'].squeeze(0),
            'attention_mask': enc['attention_mask'].squeeze(0),
            'labels':         torch.tensor(self.labels[idx], dtype=torch.long)
        }

train_ds = NewsDataset(X_raw_train, y_train, tokenizer, MAX_LEN)
val_ds   = NewsDataset(X_raw_val,   y_val,   tokenizer, MAX_LEN)
test_ds  = NewsDataset(X_raw_test,  y_test,  tokenizer, MAX_LEN)

print(f'Train dataset: {len(train_ds):,} samples')
print(f'Val dataset:   {len(val_ds):,} samples')
print(f'Test dataset:  {len(test_ds):,} samples')

# Verify one sample
sample = train_ds[0]
print(f"\nInput shape: {sample['input_ids'].shape}")

In [ ]:
from sklearn.metrics import f1_score as sk_f1

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {
        'accuracy': (preds == labels).mean(),
        'f1': sk_f1(labels, preds, average='macro')
    }

# Load model with classification head
model_xlmr = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=N_CLASSES,
    id2label=id2label,
    label2id=label2id
)

training_args = TrainingArguments(
    output_dir='./xlmr-news-uk',
    num_train_epochs=4,               
    per_device_train_batch_size=16,   
    per_device_eval_batch_size=32,
    gradient_accumulation_steps=2,    
    learning_rate=2e-5,
    lr_scheduler_type='cosine',       
    warmup_ratio=0.1,                 
    weight_decay=0.01,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='f1',
    greater_is_better=True,
    fp16=True,                      
    logging_steps=50,
    report_to='none',
    seed=SEED
)

trainer = Trainer(
    model=model_xlmr,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

print('Starting XLM-RoBERTa fine-tuning...')
train_result = trainer.train()
print('\nTraining complete!')
print(f'Train runtime: {train_result.metrics["train_runtime"]:.0f}s')

In [ ]:
# Training curve
log_history = trainer.state.log_history

train_steps = [l['step'] for l in log_history if 'loss' in l and 'eval_loss' not in l]
train_loss  = [l['loss'] for l in log_history if 'loss' in l and 'eval_loss' not in l]
eval_epochs = [l['epoch'] for l in log_history if 'eval_f1' in l]
eval_f1     = [l['eval_f1'] for l in log_history if 'eval_f1' in l]

fig, ax1 = plt.subplots(figsize=(12, 5))
ax2 = ax1.twinx()

ax1.plot(train_steps, train_loss, 'b-', alpha=0.6, label='Training Loss')
ax2.plot(eval_epochs, eval_f1, 'ro-', linewidth=2, markersize=8, label='Val Macro F1')

ax1.set_xlabel('Training Step')
ax1.set_ylabel('Training Loss', color='blue')
ax2.set_ylabel('Validation Macro F1', color='red')
ax1.set_title('XLM-RoBERTa Training Curve')

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='center right')

plt.tight_layout()
plt.savefig('training_curve.png')
plt.show()

In [ ]:
# Evaluate on test set
print('Evaluating XLM-RoBERTa on test set...')
test_predictions = trainer.predict(test_ds)
y_pred_xlmr = np.argmax(test_predictions.predictions, axis=1)

acc_xlmr = accuracy_score(y_test, y_pred_xlmr)
f1_xlmr  = f1_score(y_test, y_pred_xlmr, average='macro')

print(f'\nTest Accuracy:  {acc_xlmr:.4f}')
print(f'Test Macro F1:  {f1_xlmr:.4f}')
print('\n--- Classification Report ---')
print(classification_report(y_test, y_pred_xlmr,
                             target_names=[id2label[i] for i in range(N_CLASSES)]))

In [ ]:
# Confusion matrix - Model 3
fig, ax = plt.subplots(figsize=(9, 7))
cm_xlmr = confusion_matrix(y_test, y_pred_xlmr)
disp = ConfusionMatrixDisplay(cm_xlmr, display_labels=[id2label[i] for i in range(N_CLASSES)])
disp.plot(ax=ax, colorbar=True, cmap='Greens', xticks_rotation=30)
ax.set_title('Confusion Matrix: XLM-RoBERTa (fine-tuned)')
plt.tight_layout()
plt.savefig('cm_xlmr.png')
plt.show()

## 9. Models Comparison

In [ ]:
from sklearn.metrics import precision_score, recall_score

results = {
    'TF-IDF + LogReg': {
        'accuracy':  accuracy_score(y_test, y_pred_lr),
        'precision': precision_score(y_test, y_pred_lr,  average='macro'),
        'recall':    recall_score(y_test, y_pred_lr,    average='macro'),
        'macro_f1':  f1_score(y_test, y_pred_lr,        average='macro'),
        'weighted_f1': f1_score(y_test, y_pred_lr,      average='weighted'),
    },
    'TF-IDF + LinearSVC': {
        'accuracy':  accuracy_score(y_test, y_pred_svc),
        'precision': precision_score(y_test, y_pred_svc, average='macro'),
        'recall':    recall_score(y_test, y_pred_svc,   average='macro'),
        'macro_f1':  f1_score(y_test, y_pred_svc,       average='macro'),
        'weighted_f1': f1_score(y_test, y_pred_svc,     average='weighted'),
    },
    'XLM-RoBERTa': {
        'accuracy':  accuracy_score(y_test, y_pred_xlmr),
        'precision': precision_score(y_test, y_pred_xlmr, average='macro'),
        'recall':    recall_score(y_test, y_pred_xlmr,   average='macro'),
        'macro_f1':  f1_score(y_test, y_pred_xlmr,       average='macro'),
        'weighted_f1': f1_score(y_test, y_pred_xlmr,     average='weighted'),
    }
}

df_results = pd.DataFrame(results).T.round(4)
print('=== Final Model Comparison on Test Set ===')
print(df_results.to_string())
df_results.to_csv('model_comparison.csv')

In [ ]:
# Grouped bar chart comparison
metrics_to_plot = ['accuracy', 'precision', 'recall', 'macro_f1']
metric_labels   = ['Accuracy', 'Macro Precision', 'Macro Recall', 'Macro F1']

model_names = list(results.keys())
x = np.arange(len(metrics_to_plot))
width = 0.25
colors = ['#4878CF', '#6ACC65', '#D65F5F']

fig, ax = plt.subplots(figsize=(12, 6))

for i, (model_name, color) in enumerate(zip(model_names, colors)):
    vals = [results[model_name][m] for m in metrics_to_plot]
    bars = ax.bar(x + i*width, vals, width, label=model_name,
                  color=color, alpha=0.85, edgecolor='white')
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.003,
                f'{val:.3f}', ha='center', va='bottom', fontsize=9)

ax.set_xticks(x + width)
ax.set_xticklabels(metric_labels, fontsize=11)
ax.set_ylim(0.80, 1.00)
ax.set_ylabel('Score', fontsize=12)
ax.set_title('Model Comparison: All Metrics on Test Set', fontsize=13)
ax.legend(fontsize=10)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('model_comparison.png')
plt.show()

# Improvement analysis
print('\n--- Improvement Analysis ---')
f1_lr_val  = df_results.loc['TF-IDF + LogReg', 'macro_f1']
f1_svc_val = df_results.loc['TF-IDF + LinearSVC', 'macro_f1']
f1_xlmr_val = df_results.loc['XLM-RoBERTa', 'macro_f1']
print(f'LinearSVC vs LogReg:    +{(f1_svc_val - f1_lr_val)*100:.1f} pp')
print(f'XLM-RoBERTa vs LogReg:  +{(f1_xlmr_val - f1_lr_val)*100:.1f} pp')
print(f'XLM-RoBERTa vs LinearSVC: +{(f1_xlmr_val - f1_svc_val)*100:.1f} pp')

In [ ]:
# Per-class F1 comparison
def get_per_class_f1(y_true, y_pred):
    return f1_score(y_true, y_pred, average=None)

f1_per_class = pd.DataFrame({
    'TF-IDF + LogReg':    get_per_class_f1(y_test, y_pred_lr),
    'TF-IDF + LinearSVC': get_per_class_f1(y_test, y_pred_svc),
    'XLM-RoBERTa':        get_per_class_f1(y_test, y_pred_xlmr),
}, index=[id2label[i] for i in range(N_CLASSES)])

fig, ax = plt.subplots(figsize=(12, 6))
f1_per_class.plot(kind='bar', ax=ax, colormap='Set1', width=0.75, edgecolor='white')
ax.set_xticklabels(f1_per_class.index, rotation=25, ha='right')
ax.set_ylabel('F1-Score')
ax.set_ylim(0.7, 1.0)
ax.set_title('Per-Class F1 Score Comparison')
ax.legend(loc='lower right')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('per_class_f1_comparison.png')
plt.show()

print(f1_per_class.round(3).to_string())

## 10. Conclusions

### Key Findings

1. **XLM-RoBERTa** achieves **macro F1 = 0.9664**, outperforming the baseline
   (TF-IDF + LogReg) by **+5.6 percentage points** and LinearSVC by **+5.3 pp**.

2. **Sports (спорт)** is the easiest class due to distinctive sports-specific
   vocabulary (match names, player names, scores).

3. **News (новини) ↔ Politics (політика)** is the most challenging class pair,
   as political events are frequently covered in general news articles.

4. **TF-IDF + LinearSVC** provides a strong, fast baseline (F1 = 0.9135) that
   runs without GPU in seconds — suitable for production where latency matters.

5. All three models exceed **macro F1 = 0.91**, which reflects the high
   lexical separability of the 5-class `FIdo-AI/ua-news` dataset.

In [ ]:
# Save model and tokenizer
trainer.save_model('./best_xlmr_uk_news')
tokenizer.save_pretrained('./best_xlmr_uk_news')
print('Model saved to ./best_xlmr_uk_news')

# Save predictions
df_predictions = pd.DataFrame({
    'text': X_raw_test,
    'true_label': [id2label[i] for i in y_test],
    'pred_lr':   [id2label[i] for i in y_pred_lr],
    'pred_svc':  [id2label[i] for i in y_pred_svc],
    'pred_xlmr': [id2label[i] for i in y_pred_xlmr],
})
df_predictions['correct_xlmr'] = df_predictions['true_label'] == df_predictions['pred_xlmr']
df_predictions.to_csv('test_predictions.csv', index=False)
print('Predictions saved to test_predictions.csv')
print(f'\nFinal results summary:')
print(df_results.round(4).to_string())